# Customer Churn — Exploratory Data Analysis
**Dataset:** WA_Fn-UseC_-Telco-Customer-Churn.csv (7043 rows × 21 features)  
**Goal:** Understand churn patterns, spot class imbalance, identify top drivers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

In [ ]:
df = pd.read_csv("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 1. Basic shape
print(df.shape)          # (7043, 21)
print(df.dtypes)

In [ ]:
# 2. Churn rate (class imbalance!)
churn_rate = df["Churn"].value_counts(normalize=True)
print(churn_rate)  # No: 73.5%, Yes: 26.5% — IMBALANCED

fig, ax = plt.subplots(figsize=(5, 4))
churn_rate.plot(kind='bar', ax=ax, color=['#10b981','#ef4444'])
ax.set_title('Churn Distribution', fontsize=14)
ax.set_ylabel('Proportion')
ax.set_xticklabels(['No Churn','Churn'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 3. TotalCharges is object type (bug — has spaces)
print(df["TotalCharges"].dtype)        # object!
print(df[df["TotalCharges"]==" "])     # 11 rows with spaces

In [ ]:
# 4. Numeric feature distributions by churn
df_plot = df.copy()
df_plot["TotalCharges"] = pd.to_numeric(df_plot["TotalCharges"], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=df_plot, x=col, hue="Churn", ax=ax, bins=30, palette=['#10b981','#ef4444'])
    ax.set_title(col)
plt.suptitle('Numeric Features by Churn Status', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Categorical features — churn rate per value
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"\n--- {col} ---")
    print(df.groupby(col)["Churn"].apply(lambda x: (x=="Yes").mean()).round(3).sort_values(ascending=False))

In [ ]:
# 6. Key insight: Contract type vs Churn
fig, ax = plt.subplots(figsize=(8, 5))
contract_churn = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()).round(3)
bars = ax.bar(contract_churn.index, contract_churn.values, color=['#ef4444','#f59e0b','#10b981'])
ax.bar_label(bars, fmt='%.1%')
ax.set_title('Churn Rate by Contract Type', fontsize=14)
ax.set_ylabel('Churn Rate')
ax.set_ylim(0, 0.55)
plt.tight_layout()
plt.show()
print("\n⚠️ Key insight: Month-to-month contracts churn at 42% vs 3% for 2-year contracts!")

In [ ]:
# 7. Correlation heatmap (numeric cols)
df_num = df.copy()
df_num['TotalCharges'] = pd.to_numeric(df_num['TotalCharges'], errors='coerce')
df_num['Churn_num'] = (df_num['Churn'] == 'Yes').astype(int)
corr = df_num[['tenure','MonthlyCharges','TotalCharges','Churn_num']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, center=0)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()